In [1]:
!pip install python-docx openpyxl pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.3 MB/s eta 0:00:00


In [6]:
import os
import re
import io
import pandas as pd
import numpy as np
import docx
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from google.colab import files

# ==========================================
# 1. CARGA DE ARCHIVOS EN GOOGLE COLAB
# ==========================================
print("=== PASO 1: SUBIR LAS 3 BASES DE DATOS ===")
print("Por favor, sube los archivos de 'Base Rutex', 'Exportaciones 2024' y 'Exportaciones 2025'.")
uploaded = files.upload()

def cargar_dataframe(filename):
    """Carga un archivo en Pandas soporta .xlsx, .xls y .csv"""
    ext = os.path.splitext(filename)[1].lower()
    if ext in ['.xlsx', '.xls']:
        return pd.read_excel(filename)
    elif ext == '.csv':
        try:
            return pd.read_csv(filename, encoding='utf-8')
        except UnicodeDecodeError:
            return pd.read_csv(filename, encoding='latin1')
    else:
        raise ValueError(f"Formato no soportado: {filename}")

# Identificación automática de archivos por nombre o contenido
df_rutex = None
df_exp_2024 = None
df_exp_2025 = None

for fn in uploaded.keys():
    fn_lower = fn.lower()
    df_temp = cargar_dataframe(fn)
    cols_upper = [str(c).upper() for c in df_temp.columns]

    if 'rutex' in fn_lower or 'EVENTO' in cols_upper or 'APEX' in cols_upper:
        df_rutex = df_temp
        print(f"✔ Base Rutex identificada: {fn}")
    elif '2024' in fn_lower or ('AÑO' in cols_upper and (df_temp['Año' if 'Año' in df_temp.columns else 'AÑO'] == 2024).any()):
        df_exp_2024 = df_temp
        print(f"✔ Exportaciones 2024 identificadas: {fn}")
    elif '2025' in fn_lower or ('AÑO' in cols_upper and (df_temp['Año' if 'Año' in df_temp.columns else 'AÑO'] == 2025).any()):
        df_exp_2025 = df_temp
        print(f"✔ Exportaciones 2025 identificadas: {fn}")

# Verificación de que los 3 archivos fueron cargados
if df_rutex is None or df_exp_2024 is None or df_exp_2025 is None:
    raise ValueError("Error: No se pudieron identificar las 3 bases requeridas. Revisa los nombres de tus archivos.")

# ==========================================
# 2. INGRESO DEL PAÍS POR PARTE DEL USUARIO
# ==========================================
print("\n" + "="*50)
pais_usuario = input("Ingrese el país que requiere consultar (ej: Uruguay): ").strip()

# ==========================================
# 3. LIMPIEZA Y ESTANDARIZACIÓN DE DATOS
# ==========================================
def limpiar_ruc(series):
    """Limpia la columna RUC para asegurar consistencia"""
    return series.astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

def limpiar_monto(series):
    """Convierte importes a numérico"""
    return pd.to_numeric(series.astype(str).str.replace(',', '').str.strip(), errors='coerce').fillna(0)

# Estandarizar RUC y montos
df_rutex['RUC_CLEAN'] = limpiar_ruc(df_rutex['RUC'])
df_exp_2024['RUC_CLEAN'] = limpiar_ruc(df_exp_2024['RUC'])
df_exp_2025['RUC_CLEAN'] = limpiar_ruc(df_exp_2025['RUC'])

df_exp_2024['FOB USD'] = limpiar_monto(df_exp_2024['FOB USD'])
df_exp_2025['FOB USD'] = limpiar_monto(df_exp_2025['FOB USD'])

# Filtrar exportaciones por País
mask_2024 = df_exp_2024['MERCADO'].astype(str).str.strip().str.upper() == pais_usuario.upper()
mask_2025 = df_exp_2025['MERCADO'].astype(str).str.strip().str.upper() == pais_usuario.upper()

df_2024_p = df_exp_2024[mask_2024]
df_2025_p = df_exp_2025[mask_2025]

# Universos de RUCs únicos de Rutex
rucs_rutex = set(df_rutex['RUC_CLEAN'].unique())

# Cruzar exportaciones con la base Rutex para quedarse solo con empresas de Rutex
df_2024_p_rutex = df_2024_p[df_2024_p['RUC_CLEAN'].isin(rucs_rutex)]
df_2025_p_rutex = df_2025_p[df_2025_p['RUC_CLEAN'].isin(rucs_rutex)]

# Filtrar Rutex para empresas exportadoras a ese país en el 2025
rucs_exportadores_2025_rutex = set(df_2025_p_rutex['RUC_CLEAN'].unique())
df_rutex_p = df_rutex[df_rutex['RUC_CLEAN'].isin(rucs_exportadores_2025_rutex)]

# ==========================================
# 4. CÁLCULO DE MÉTRICAS Y ESTADÍSTICAS
# ==========================================
# 1. Número de empresas de Rutex que exportaron a ese país en 2025
cant_empresas_rutex = df_rutex_p['RUC_CLEAN'].nunique()

# 2. Número de partidas únicas (productos) exportados en 2025 por empresas Rutex
cant_productos_rutex = df_2025_p_rutex['PARTIDA_DESCRIP_LARGA'].nunique()

# 3. Montos exportados por empresas Rutex en 2024 y 2025 (en USD Millones)
fob_2024_total = df_2024_p_rutex['FOB USD'].sum()
fob_2025_total = df_2025_p_rutex['FOB USD'].sum()

fob_2024_millones = fob_2024_total / 1_000_000
fob_2025_millones = fob_2025_total / 1_000_000

# 4. Variación Porcentual
if fob_2024_total > 0:
    var_pct = ((fob_2025_total - fob_2024_total) / fob_2024_total) * 100
else:
    var_pct = 0.0

tipo_var = "disminución" if var_pct < 0 else "incremento"
abs_var_pct = abs(var_pct)

# Función para formatear valores monetarios a texto estilo informe
def formatear_monto_texto(val):
    if val >= 1_000_000:
        v_m = val / 1_000_000
        return f"US$ {int(v_m)} millones" if v_m.is_integer() else f"US$ {v_m:.1f} millones"
    elif val >= 1_000:
        v_k = val / 1_000
        return f"USD {int(v_k)} mil" if v_k.is_integer() else f"USD {v_k:.0f} mil"
    else:
        return f"USD {int(val)}"

# Función para limpiar el código de la partida del nombre del producto
def limpiar_codigo_producto(nombre_producto):
    # Reemplaza cualquier número inicial seguido de un guion opcional y espacios
    return re.sub(r'^\d+\s*-\s*', '', str(nombre_producto)).strip()

# 5. Principales productos exportados en 2025 (Top 5) - Solo Cruce Rutex
top_productos = (
    df_2025_p_rutex.groupby('PARTIDA_DESCRIP_LARGA')['FOB USD']
    .sum()
    .reset_index()
    .sort_values(by='FOB USD', ascending=False)
    .head(5)
)

prod_textos = []
for _, row in top_productos.iterrows():
    p_nom_limpio = limpiar_codigo_producto(row['PARTIDA_DESCRIP_LARGA']).lower()
    p_monto = formatear_monto_texto(row['FOB USD'])
    prod_textos.append(f"{p_monto} en {p_nom_limpio}")

texto_productos = ", ".join(prod_textos) if prod_textos else "sin registros de productos"

# 6. Principales regiones exportadoras en 2025 (Top 5) - Extraído de la columna DEPARTAMENTO de la base Exportaciones 2025 cruzada con Rutex
top_regiones = (
    df_2025_p_rutex.groupby('DEPARTAMENTO')
    .agg(fob=('FOB USD', 'sum'), empresas=('RUC_CLEAN', 'nunique'))
    .reset_index()
    .sort_values(by='fob', ascending=False)
    .head(5)
)

reg_textos = []
for _, row in top_regiones.iterrows():
    r_nom = str(row['DEPARTAMENTO']).strip().title()
    r_monto = formatear_monto_texto(row['fob'])
    r_emp = row['empresas']
    emp_word = "empresa" if r_emp == 1 else "empresas"
    reg_textos.append(f"{r_nom} ({r_monto} / {r_emp} {emp_word})")

texto_regiones = ", ".join(reg_textos) if reg_textos else "sin registros regionales"

# 7. Principales programas/eventos en Rutex para estas empresas (Top 15)
top_eventos = (
    df_rutex_p.groupby('EVENTO')['RUC_CLEAN']
    .nunique()
    .reset_index()
    .rename(columns={'RUC_CLEAN': 'CANT_EMPRESAS'})
    .sort_values(by='CANT_EMPRESAS', ascending=False)
    .head(15)
)

# ==========================================
# 5. GENERACIÓN DEL DOCUMENTO WORD (.DOCX)
# ==========================================
doc = docx.Document()

# Configurar márgenes de página
sections = doc.sections
for section in sections:
    section.top_margin = Inches(1)
    section.bottom_margin = Inches(1)
    section.left_margin = Inches(1)
    section.right_margin = Inches(1)

# Título Principal
p_title = doc.add_paragraph()
r_title = p_title.add_run("Ayuda memoria")
r_title.bold = True
r_title.font.name = 'Calibri'
r_title.font.size = Pt(20)
r_title.font.color.rgb = RGBColor(0, 51, 102)

# Subtítulo
p_sub = doc.add_paragraph()
r_sub = p_sub.add_run(f"Empresas exportadoras a {pais_usuario.capitalize()} - Ruta Exportadora")
r_sub.bold = True
r_sub.font.name = 'Calibri'
r_sub.font.size = Pt(14)
r_sub.font.color.rgb = RGBColor(80, 80, 80)

# Párrafo 1: Estadísticas Generales (Modificado para incluir número de partidas/productos)
p1 = doc.add_paragraph()
p1.paragraph_format.line_spacing = 1.15
p1.paragraph_format.space_after = Pt(8)

t1 = (
    f"Durante el 2026 se atendieron a {cant_empresas_rutex} empresas exportadoras que atienden al mercado de {pais_usuario.capitalize()}, "
    f"a través de {cant_productos_rutex} productos (número de partidas únicas), el monto exportado en el 2025 (enero-diciembre) "
    f"fue de USD {fob_2025_millones:.1f} millones y en el 2024 fue USD {fob_2024_millones:.1f} millones, con lo que se observó "
    f"una {tipo_var} de {abs_var_pct:.2f}% con respecto al año anterior."
)
r_p1 = p1.add_run(t1)
r_p1.font.name = 'Calibri'
r_p1.font.size = Pt(11)

# Párrafo 2: Productos y Regiones
p2 = doc.add_paragraph()
p2.paragraph_format.line_spacing = 1.15
p2.paragraph_format.space_after = Pt(12)

t2 = (
    f"Los principales productos exportados en el 2025 fueron {texto_productos}. "
    f"Las principales regiones exportadoras son {texto_regiones}."
)
r_p2 = p2.add_run(t2)
r_p2.font.name = 'Calibri'
r_p2.font.size = Pt(11)

# Párrafo Encabezado de Eventos
p_ev_head = doc.add_paragraph()
r_ev_head = p_ev_head.add_run("Los principales programas en donde participaron las empresas fueron:")
r_ev_head.bold = True
r_ev_head.font.name = 'Calibri'
r_ev_head.font.size = Pt(11)
p_ev_head.paragraph_format.space_after = Pt(6)

# Viñetas Top 15 Eventos
if not top_eventos.empty:
    for idx, row in top_eventos.iterrows():
        p_item = doc.add_paragraph(style='List Bullet')
        p_item.paragraph_format.space_after = Pt(3)
        p_item.paragraph_format.line_spacing = 1.15

        evento_nombre = str(row['EVENTO']).strip()
        num_emp = row['CANT_EMPRESAS']
        emp_label = "empresa participante" if num_emp == 1 else "empresas participantes"

        r_item = p_item.add_run(f"{evento_nombre}: {num_emp} {emp_label}")
        r_item.font.name = 'Calibri'
        r_item.font.size = Pt(11)
else:
    p_item = doc.add_paragraph(style='List Bullet')
    r_item = p_item.add_run("No se registraron eventos o programas para las empresas exportadoras a este destino.")
    r_item.font.name = 'Calibri'

# ==========================================
# 6. GUARDAR Y DESCARGAR EL ARCHIVO WORD
# ==========================================
nombre_archivo = f"Ayuda_Memoria_Exportaciones_{pais_usuario.replace(' ', '_')}.docx"
doc.save(nombre_archivo)

print("\n" + "="*50)
print(f"✔ Documento '{nombre_archivo}' generado exitosamente.")
print("Descargando archivo Word...")
files.download(nombre_archivo)

=== PASO 1: SUBIR LAS 3 BASES DE DATOS ===
Por favor, sube los archivos de 'Base Rutex', 'Exportaciones 2024' y 'Exportaciones 2025'.


Saving base rutex 29.09.2026 - Colab.xlsx to base rutex 29.09.2026 - Colab (3).xlsx
Saving EXPORTACIONES 2024.xlsx to EXPORTACIONES 2024 (3).xlsx
Saving EXPORTACIONES 2025.xlsx to EXPORTACIONES 2025 (3).xlsx
✔ Base Rutex identificada: base rutex 29.09.2026 - Colab (3).xlsx
✔ Exportaciones 2024 identificadas: EXPORTACIONES 2024 (3).xlsx
✔ Exportaciones 2025 identificadas: EXPORTACIONES 2025 (3).xlsx

Ingrese el país que requiere consultar (ej: Uruguay): Uruguay

✔ Documento 'Ayuda_Memoria_Exportaciones_Uruguay.docx' generado exitosamente.
Descargando archivo Word...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>